# Wordle openers and spoiler-safe difficulty: a starter notebook

This notebook loads the **FiveLetterWords Wordle Research Corpus (v2026-09-01)** and draws three quick charts:

1. The best familiar opening words by average guesses
2. First-turn information (entropy) vs six-turn performance for all 14,855 accepted guesses
3. How the mix of solver-pressure labels has changed by year (no answer words involved)

**Dataset:**

- Kaggle: [kaggle.com/datasets/fiveletterwords/fiveletterwords-wordle-research](https://www.kaggle.com/datasets/fiveletterwords/fiveletterwords-wordle-research)
- Zenodo DOI: [10.5281/zenodo.23168432](https://doi.org/10.5281/zenodo.23168432)
- Canonical home: [https://www.fiveletterwords.io/data](https://www.fiveletterwords.io/data) (release page: https://www.fiveletterwords.io/data/releases/v2026-09-01/)

Licensed CC BY 4.0. Please credit FiveLetterWords.io.

*Independent research. Not affiliated with or endorsed by The New York Times. Wordle is a trademark of The New York Times Company.*

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_data_dir():
    # Kaggle mounts datasets under /kaggle/input; locally use FLW_DATA_DIR or the unzipped package.
    candidates = []
    if os.environ.get("FLW_DATA_DIR"):
        candidates.append(Path(os.environ["FLW_DATA_DIR"]))
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        candidates += [p.parent for p in kaggle.glob("**/opener-rankings.csv")]
    candidates += [Path("data"), Path("../fiveletterwords-wordle-research-v2026-09-01/data")]
    for c in candidates:
        if (c / "opener-rankings.csv").exists():
            return c
    raise FileNotFoundError("Could not find opener-rankings.csv; attach the dataset or set FLW_DATA_DIR.")

DATA = find_data_dir()
print("Using data from:", DATA)
plt.rcParams.update({"figure.figsize": (10, 5.5), "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
openers = pd.read_csv(DATA / "opener-rankings.csv")
familiar = pd.read_csv(DATA / "familiar-opener-benchmark.csv")
difficulty = pd.read_csv(DATA / "puzzle-difficulty.csv", parse_dates=["date"])

print(f"opener-rankings: {len(openers):,} rows | familiar-opener-benchmark: {len(familiar):,} rows | puzzle-difficulty: {len(difficulty):,} rows")
familiar.head()

## 1. Best familiar openers

`familiar-opener-benchmark` filters the full benchmark to everyday vocabulary. `average_guesses` comes from a fixed-policy six-turn simulation over 2,354 answers (solved games only), so lower is better. These are policy-dependent benchmarks, not proofs of optimal play.

In [ ]:
top = familiar.nsmallest(15, "average_guesses").iloc[::-1]
fig, ax = plt.subplots()
ax.barh(top["word"].str.upper(), top["average_guesses"], color="#6aaa64")
ax.set_xlim(top["average_guesses"].min() - 0.03, top["average_guesses"].max() + 0.02)
for y, v in enumerate(top["average_guesses"]):
    ax.text(v + 0.002, y, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("Average guesses to solve (lower is better)")
ax.set_title("Top 15 familiar Wordle openers, FiveLetterWords v2026-09-01")
plt.tight_layout()
plt.show()

## 2. Does first-turn information predict the whole game?

`entropy_bits` measures how evenly a first guess splits the 2,373-answer working pool. Higher entropy usually means fewer guesses overall, but the relationship is not perfect: the six-turn simulation uses a different answer pool and a fixed continuation policy.

In [ ]:
fig, ax = plt.subplots()
cand = openers["is_answer_candidate"].astype(str).str.lower() == "true"
ax.scatter(openers.loc[~cand, "entropy_bits"], openers.loc[~cand, "average_guesses"], s=4, alpha=0.25, color="#787c7e", label="Guess-only words")
ax.scatter(openers.loc[cand, "entropy_bits"], openers.loc[cand, "average_guesses"], s=4, alpha=0.4, color="#c9b458", label="Possible answers")
for w, offset in {"slant": (-10, -14), "crane": (-45, 10), "slate": (6, 8), "adieu": (6, 6), "audio": (-40, 8)}.items():
    r = openers.loc[openers["word"] == w]
    if not r.empty:
        ax.annotate(w.upper(), (r["entropy_bits"].iat[0], r["average_guesses"].iat[0]), xytext=offset, textcoords="offset points", fontsize=9, weight="bold", arrowprops={"arrowstyle": "-", "lw": 0.6})
corr = openers["entropy_bits"].corr(openers["average_guesses"])
ax.set_xlabel("First-turn entropy (bits)")
ax.set_ylabel("Average guesses (six-turn simulation)")
ax.set_title(f"Entropy vs average guesses for all {len(openers):,} accepted guesses (r = {corr:.2f})")
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

## 3. Solver pressure by year (spoiler-safe)

`puzzle-difficulty` has one row per completed puzzle with **no answer words**. `solver_pressure_percentile` ranks each puzzle within this release using reference-opener remainders, replay-path length, trap-family size and repeated letters. It is a model comparison, **not** a measure of how hard humans found the puzzle.

In [ ]:
order = ["Easier for the solver", "Middle of the pack", "Tricky", "Hard for the solver"]
colours = ["#6aaa64", "#a8c99c", "#c9b458", "#787c7e"]
by_year = (pd.crosstab(difficulty["date"].dt.year, difficulty["solver_pressure_label"], normalize="index")[order] * 100)
counts = difficulty["date"].dt.year.value_counts().sort_index()

ax = by_year.plot(kind="bar", stacked=True, color=colours, width=0.75)
ax.set_xticklabels([f"{y}\n(n={counts[y]})" for y in by_year.index], rotation=0)
ax.set_xlabel("")
ax.set_ylabel("% of completed puzzles")
ax.set_ylim(0, 100)
ax.set_title("Solver-pressure labels by year (spoiler-safe; 2021 and 2026 are partial years)")
ax.legend(title="Solver pressure", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

by_year.round(1)

## Next steps

- Explore `trap-families.csv` for one-slot families such as `_IGHT` and their best probe words.
- Compare opening pairs in `opening-pair-frontier.csv`.
- Use `wordle-2000-fact-sheet.csv` for milestone facts ahead of Wordle #2000.
- Field definitions are in `data-dictionary.csv`; methods in `METHODS.md`.

**Please cite:** FiveLetterWords.io (2026-09-01). FiveLetterWords research release v2026-09-01 [Data set]. Zenodo. https://doi.org/10.5281/zenodo.23168432 (CC BY 4.0)

Dataset on Kaggle: https://www.kaggle.com/datasets/fiveletterwords/fiveletterwords-wordle-research · More FiveLetterWords research data: https://www.fiveletterwords.io/data